In [ ]:
@cached_function
def RR(k):
    ring = PolynomialRing(QQ, names=['q', 't', 'u', 'v'] + ['y%d' % (i,) for i in range(k)])
    return ring.fraction_field()


@cached_function
def VV(k):
    return SymmetricFunctions(RR(k))

Symqt = VV(0)
q = RR(0).gen(0)
t = RR(0).gen(1)
u = RR(0).gen(2)
v = RR(0).gen(3)


@cached_function
def qq(k=0):
    return RR(k).gen(0)


@cached_function
def tt(k=0):
    return RR(k).gen(1)


@cached_function
def uu(k=0):
    return RR(k).gen(2)


@cached_function
def vv(k=0):
    return RR(k).gen(3)


@cached_function
def yy(i, k=None):
    if k is None:
        k = i + 1
    return RR(k).gen(i+4)


@cached_function
def XX(k=0):
    return VV(k).monomial()([1])


@cached_function
def XX0(k=0):
    return VV(k).one()


def space_index(f):
    if f == 0:
        return 0
    return len(f.parent().base_ring().gens()) - 4


def delta0(f, x, y, k=None):
    if k is None:
        k = space_index(f)
    return ((qq(k)-1)*y*f + (y-qq(k)*x)*f.subs({x: y, y: x}))/(qq(k)*(y-x))


def deltastar0(f, x, y, k=None):
    if k is None:
        k = space_index(f)
    return ((qq(k)-1)*x*f + (y-qq(k)*x)*f.subs({x: y, y: x}))/(y-x)


def act_on_coefficients(operator, f, k):
    return sum(VV(k).monomial()(mu) * operator(cf) for (mu, cf) in VV(k).monomial()(f))


def TT(f, i, k=None):
    if k is None:
        k = space_index(f)
    return act_on_coefficients(lambda g: deltastar0(g, yy(i, k), yy(i+1, k), k), f, k)


def TTstar(f, i, k=None):
    if k is None:
        k = space_index(f)
    return act_on_coefficients(lambda g: delta0(g, yy(i, k), yy(i+1, k), k), f, k)


def dplus(f, k=None, power=1, old=False):
    if k is None:
        k = space_index(f)
    if f == 0:
        return 0
    if power == 0:
        return f
    f1 = sum(
        cf(RR(k + 1).gens()[:-1]) * VV(k + 1).monomial()(p)
        for p, cf in VV(k).monomial()(f)
    )
    f = f1(XX(k+1)+(qq(k+1)-1)*yy(k, k+1)*XX0(k+1))
    if not old:
        f *= -yy(k, k+1)
        for i in range(k):
            f = TT(f, k-i-1, k+1)
    return dplus(f, k=k+1, power=power-1, old=old)


def dequal(f, k=None):
    if k is None:
        k = space_index(f)
    for i in range(k-1):
        f = TTstar(f, k-i-2, k)
    return f * yy(0, k)


def dplusstar(f, k=None, power=1):
    if k is None:
        k = space_index(f)
    if power == 0:
        return f
    substitutions = [qq(k+1), tt(k+1), uu(k+1), vv(k+1)] + [yy(i+1, k+1) for i in range(k)] + [tt(k+1)*yy(0, k+1)]
    return dplusstar(act_on_coefficients(lambda g: g(substitutions), dplus(f, k, power=1, old=True), k+1), 
        k=k+1, power=power-1)


def dplusnabla(f, k):
    return -dplusstar(f, k)*qq(k+1)**k


def dminus(f, k=None, power=1, old=False):
    k = space_index(f)
    if power == 0:
        return f
    eps = 1 if old else 0
    f = f(XX(k) - (qq(k)-1) * yy(k-1, k) * XX0(k))
    out = 0
    for (mu, cf) in VV(k).monomial()(f):
        RT = RR(k-1)['T']
        T = RT.gens()[0]
        num = RT(RR(k)(cf).numerator()(list(RR(k-1).gens())+[T]))
        den = RT(RR(k)(cf).denominator()(list(RR(k-1).gens())+[T]))
        assert len(den.coefficients()) == 1
        if num.degree() < den.degree():
            continue
        out += sum(num[i+den.degree()]/den(T=1) * VV(k-1).monomial()[[1]*(i+eps)]*(-1)**i 
                   for i in range(num.degree()-den.degree()+1)) * VV(k-1).monomial()(mu)
    return dminus(out, k=k-1, power=power-1, old=old)


def dcomm(f, k=None, weight=0):
    if k is None:
        k = space_index(f)
    return ((1 - weight) * dminus(dplus(f, k), k+1) + (q * weight - 1) * dplus(dminus(f, k), k-1))/(qq(k)-1)


def dword(w, f=XX0(0)):
    # w = w[::-1]
    for (i,x) in enumerate(w):
        if x == 1:
            f = dplus(f, sum(w[:i]))
        elif x == 0:
            f = dcomm(f, sum(w[:i]))
        elif x == -1:
            f = dminus(f, sum(w[:i]))
        else:
            raise ValueError(f"Invalid value in word: {x}. Expected 1, 0, or -1.")
    return f


def zz1(f, k=None, power=1):
    if k is None:
        k = space_index(f)
    if power > 1:
        return zz1(f, k, power-1)
    if f == 0:
        return 0
    for i in range(k-1):
        f = TTstar(f, k, i)
    return (q**k / (1-q))*(dplusstar(dminus(f, k), k-1) - dminus(dplusstar(f, k), k+1))

In [ ]:
def B_mu(mu):
    return sum(q**c[1] * t**c[0] for c in mu.cells())


def Delta(f, g, power=1):
    if g == 0:
        return 0
    g = g*Symqt.one()
    return Symqt.schur()(sum(cf*((f(B_mu(mu)*Symqt.one()).scalar(Symqt.one()))**power)*Symqt.macdonald().Ht()(mu) for (mu, cf) in Symqt.macdonald().Ht()(g)))


def D0(f):
    return f - (1-q)*(1-t)*Delta(Symqt.schur()[1], f)


def Dn(n, f=XX0(0)):
    if n == 0:
        return D0(f)
    elif n > 0:
        return (-(1-q)*(1-t))**(-n) * sum((-1)**r * binomial(n,r) * Symqt.schur()[1]**r * D0(Symqt.schur()[1]**(n-r)*f) for r in range(n+1))
    else:
        return sum((-1)**r * binomial(-n,r) * D0(f.skew_by(Symqt.schur()[1]**r)).skew_by(Symqt.schur()[1]**(-n-r)) for r in range(-n+1))


def D_new(alpha, f=XX0(0)):
    def d_inner(alpha0, f0):
        if len(alpha0) == 0:
            return f0
        elif len(alpha0) == 1:
            return (-yy(0,1))**alpha0[0] * f0
        return d_inner(alpha0[:-1], (dplusstar(dminus((-yy(0,1))**(alpha0[-1]) * f0, 1), 0) + zz1((-yy(0,1))**(alpha0[-1]) * f0, 1)))
    return dminus(d_inner(alpha, dplusstar(f, 0)), 1)
    

def D_old(beta, f=None):
    if f is None:
        f = Symqt.one()
    if f == 0:
        return 0
    beta = tuple(beta)
    if not beta:
        return f
    elif len(beta) == 1:
        return Dn(beta[0], f)
    elif beta[-1] == -(f*Symqt.one()).degree():
        return D_old(beta[:-1], Dn(beta[-1], f))
    else:
        return D_old(beta[:-1], Dn(beta[-1], f)) + q*t*D_old(beta[:-2] + (beta[-2]+1,) + (beta[-1]-1,), f)

In [ ]:
import signal
import time


class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    raise TimeoutException()


def run_benchmarks(timeout_seconds=300):
    # Set signal handler for the 5-minute timeout
    signal.signal(signal.SIGALRM, timeout_handler)

    # Part A: Varying length & entry types (zeros, negative integers, larger entries), fixed input F = 1
    part_a = [
        ((3,), Symqt.one(), "1"),
        ((2, 1), Symqt.one(), "1"),
        ((0, 3), Symqt.one(), "1"),
        ((1, 1, 2), Symqt.one(), "1"),
        ((2, -1, 2), Symqt.one(), "1"),
        ((1, 2, 1, 1), Symqt.one(), "1"),
        ((3, 0, -1, 2), Symqt.one(), "1"),
        ((1, 1, 3, 0, 2), Symqt.one(), "1"),
        ((1, -2, 3, 0, 1), Symqt.one(), "1"),
        ((2, 0, -1, 3, -1, 2), Symqt.one(), "1"),
    ]

    # Part B: Small compositions containing zeros and negative entries, varying non-trivial inputs F
    part_b = [
        ((0, 2), Symqt.schur()[1], "s_1"),
        ((0, 2), Symqt.schur()[2], "s_2"),
        ((2, -1, 1), Symqt.schur()[1], "s_1"),
        ((2, -1, 1), Symqt.schur()[2, 1], "s_{2,1}"),
        ((3, -1, 2), Symqt.schur()[2], "s_2"),
        ((3, -1, 2), Symqt.schur()[2, 2], "s_{2,2}"),
        ((1, 0, -1, 2), Symqt.schur()[2, 1], "s_{2,1}"),
    ]

    print(
        "=== PART A: Fixed Input F = 1, Varying Composition Length & Entry"
        " Types ==="
    )
    print(
        f"{'Composition':<24} | {'Length':<6} | {'Input F':<10} |"
        f" {'D_new (s)':<12} | {'D_old (s)':<18} | {'Speedup':<8} |"
        f" {'Match?':<6}"
    )
    print("-" * 96)

    for alpha, f, f_str in part_a:
        # Time D_new (direct A_{q,t} operator action)
        t0 = time.perf_counter()
        res_alpha = D_new(alpha, f)
        t_alpha = time.perf_counter() - t0

        # Time D_old with 5-minute timeout
        signal.alarm(timeout_seconds)
        try:
            t0 = time.perf_counter()
            res_beta = D_old(alpha, f)
            t_beta = time.perf_counter() - t0
            signal.alarm(0)  # Cancel alarm on success

            match = Symqt.schur()(res_alpha) == Symqt.schur()(res_beta)
            match_str = "OK" if match else "FAIL"
            speedup_str = f"{t_beta / t_alpha:.1f}x" if t_alpha > 0 else "N/A"
            t_beta_str = f"{t_beta:.4f}"
        except TimeoutException:
            signal.alarm(0)
            t_beta_str = f"> {timeout_seconds}s (Timeout)"
            speedup_str = "N/A"
            match_str = "N/A"

        print(
            f"{str(alpha):<24} | {len(alpha):<6} | {f_str:<10} |"
            f" {t_alpha:<12.4f} | {t_beta_str:<18} | {speedup_str:<8} |"
            f" {match_str:<6}"
        )

    print(
        "\n=== PART B: Small Mixed Compositions (With Zeros/Negative"
        " Entries), Varying Input Functions ==="
    )
    print(
        f"{'Composition':<24} | {'Length':<6} | {'Input F':<10} |"
        f" {'D_new (s)':<12} | {'D_old (s)':<18} | {'Speedup':<8} |"
        f" {'Match?':<6}"
    )
    print("-" * 96)

    for alpha, f, f_str in part_b:
        t0 = time.perf_counter()
        res_alpha = D_new(alpha, f)
        t_alpha = time.perf_counter() - t0

        signal.alarm(timeout_seconds)
        try:
            t0 = time.perf_counter()
            res_beta = D_old(alpha, f)
            t_beta = time.perf_counter() - t0
            signal.alarm(0)  # Cancel alarm on success

            match = Symqt.schur()(res_alpha) == Symqt.schur()(res_beta)
            match_str = "OK" if match else "FAIL"
            speedup_str = f"{t_beta / t_alpha:.1f}x" if t_alpha > 0 else "N/A"
            t_beta_str = f"{t_beta:.4f}"
        except TimeoutException:
            signal.alarm(0)
            t_beta_str = f"> {timeout_seconds}s (Timeout)"
            speedup_str = "N/A"
            match_str = "N/A"

        print(
            f"{str(alpha):<24} | {len(alpha):<6} | {f_str:<10} |"
            f" {t_alpha:<12.4f} | {t_beta_str:<18} | {speedup_str:<8} |"
            f" {match_str:<6}"
        )


In [ ]:
dplus(XX0(0), power=6) # This is to build V(k) for k up to 6

y0*y1*y2*y3*y4*y5*m[]

In [10]:
run_benchmarks()

=== PART A: Fixed Input F = 1, Varying Composition Length & Entry Types ===
Composition              | Length | Input F    | D_new (s)  | D_old (s)         | Speedup  | Match?
------------------------------------------------------------------------------------------------
(3,)                     | 1      | 1          | 0.0030       | 0.0294             | 9.8x     | OK    
(2, 1)                   | 2      | 1          | 0.0117       | 0.0592             | 5.1x     | OK    


(0, 3)                   | 2      | 1          | 0.0259       | 0.1290             | 5.0x     | OK    


(1, 1, 2)                | 3      | 1          | 0.0604       | 0.9108             | 15.1x    | OK    


(2, -1, 2)               | 3      | 1          | 0.0375       | 0.3093             | 8.2x     | OK    


(1, 2, 1, 1)             | 4      | 1          | 0.1148       | 6.0006             | 52.3x    | OK    


...

(2, -1, 1)               | 3      | s_1        | 0.0542       | 0.2270             | 4.2x     | OK    


(2, -1, 1)               | 3      | s_{2,1}    | 0.2870       | 4.2673             | 14.9x    | OK    


(3, -1, 2)               | 3      | s_2        | 0.2290       | 12.2241            | 53.4x    | OK    


(3, -1, 2)               | 3      | s_{2,2}    | 0.9013       | 226.7809           | 251.6x   | OK    


(1, 0, -1, 2)            | 4      | s_{2,1}    | 0.9961       | 36.4460            | 36.6x    | OK    
